In [ ]:
# Cell 1: Install deps and login to HF (optional, same as before)
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
from huggingface_hub import login
login(add_to_git_credential=True)
%pip install -q datasets


In [ ]:
# Cell 2: Load TheFinAI/flare-fpb test set and normalize labels

from datasets import load_dataset, Dataset

LABELS = ["negative", "neutral", "positive"]

ds_raw = load_dataset("TheFinAI/flare-fpb", split="test")
print("Loaded flare-fpb test:", len(ds_raw), "columns:", ds_raw.column_names)

_alias = {
    "pos": "positive",
    "neg": "negative",
    "neu": "neutral",
    "bullish": "positive",
    "bearish": "negative",
}

def _norm_label(v):
    if v is None:
        return None
    if isinstance(v, (int, float)) or (isinstance(v, str) and v.isdigit()):
        i = int(v)
        return LABELS[i] if 0 <= i < len(LABELS) else None
    s = str(v).strip().lower()
    s = _alias.get(s, s)
    return s if s in LABELS else None

def _map_row(x):
    text = x.get("text") or x.get("sentence") or x.get("content") or x.get("input") or ""
    lab  = _norm_label(x.get("label", x.get("labels", x.get("answer"))))
    return {"text": text, "choices": LABELS, "answer": lab}

ds = Dataset.from_list([{**r, **_map_row(r)} for r in ds_raw])
bad = [i for i, r in enumerate(ds) if r["answer"] not in LABELS]
print("Samples with unusable label:", len(bad))
assert len(bad) == 0, "Found unparseable labels; please check the field mapping."


In [ ]:
# Cell 3: Install Claude / evaluation deps

%pip install -q "anthropic>=0.40.0" "httpx==0.27.2" "httpcore==1.0.5" "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"


In [ ]:
# Cell 4: Global config, metadata (Claude version)

import os, getpass, json, time, platform
from importlib.metadata import version, PackageNotFoundError

try:
    LABELS
except NameError:
    LABELS = ["negative", "neutral", "positive"]

# Choose a Claude model; adjust as needed
MODEL    = "claude-3-5-sonnet-20241022"
# Anthropic official base URL for Messages API
BASE_URL = "https://api.anthropic.com"

api_key = os.getenv("ANTHROPIC_API_KEY") or os.getenv("API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your Anthropic API key: ")
os.environ["ANTHROPIC_API_KEY"] = api_key

run_tag   = f"flare_fpb_claude_{MODEL.replace('/', '_')}"
save_dir  = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
meta_path = f"{save_dir}/{run_tag}_metadata.json"

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

meta = {
    "dataset": "TheFinAI/flare-fpb",
    "split": "test",
    "labels": list(LABELS),
    "model": MODEL,
    "anthropic_sdk": ver("anthropic"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "Claude Messages API; strict text-JSON protocol; retries+checkpoint",
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL, "| BASE_URL:", BASE_URL)
print("ANTHROPIC_API_KEY is set:", bool(os.environ.get("ANTHROPIC_API_KEY")))


In [ ]:
# Cell 5: Claude calling helper + main loop

import requests, json, os, re, time
import pandas as pd
from tqdm import tqdm

from anthropic import Anthropic, APIStatusError

client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_user_text(sentence: str, choices=("negative", "neutral", "positive")):
    # Keep prompt very close to your original protocol
    return (
        "Task: classify the sentence into exactly one of these labels: "
        f"{', '.join(choices)}.\n\n"
        f"Sentence: {sentence}\n\n"
        "Return ONLY a JSON object on a single line, exactly in this form:\n"
        "{\"label\":\"negative|neutral|positive\"}\n"
        "No code fences, no extra text, no explanation."
    )

def ask_claude_textjson_once(sentence, choices=("negative","neutral","positive"), max_tok=128):
    user_text = _make_user_text(sentence, choices)

    try:
        resp = client.messages.create(
            model=MODEL,
            max_tokens=max_tok,
            temperature=0,
            messages=[
                {
                    "role": "user",
                    "content": user_text,
                }
            ],
        )
    except APIStatusError as e:
        # Keep a similar error surface to your OpenAI version
        raise RuntimeError(f"Claude API error {e.status_code}: {e.message}") from e
    except Exception as e:
        raise RuntimeError(f"Claude API call failed: {type(e).__name__}: {e}") from e

    # Extract text from Claude messages API
    text_chunks = []
    for part in resp.content:
        if part.type == "text" and isinstance(part.text, str):
            text_chunks.append(part.text)
    txt = " ".join(text_chunks).strip()

    if not txt:
        raise RuntimeError(f"No text in Claude response. Snippet: {resp}")

    txt = _strip_code_fences(txt)

    try:
        obj = json.loads(txt)
    except json.JSONDecodeError:
        raise RuntimeError(f"Invalid JSON from Claude: {txt!r}")

    lab = obj.get("label")
    if lab not in choices:
        raise RuntimeError(f"Invalid label {lab!r}; raw json: {obj}")
    return lab

def ask_claude_textjson(sentence, choices=("negative","neutral","positive")):
    # Similar retry pattern as your o3 wrapper
    for max_tok in (128, 256, 512):
        delay = 1.0
        for attempt in range(5):
            try:
                return ask_claude_textjson_once(sentence, choices, max_tok=max_tok)
            except RuntimeError as e:
                msg = str(e)
                # Simple backoff on 429 / 5xx-like messages
                if "429" in msg or "Too Many Requests" in msg or "rate_limit" in msg:
                    time.sleep(delay); delay = min(delay * 2, 60); continue
                if "500" in msg or "503" in msg or "server_error" in msg:
                    time.sleep(delay); delay = min(delay * 2, 60); continue
                if "max_tokens" in msg:
                    # escalate to higher max_tok
                    break
                # otherwise, treat as fatal for this example
                raise
    raise RuntimeError("Exhausted retries and token budgets for this sample.")

run_tag   = f"flare_fpb_claude_{MODEL.replace('/', '_')}"
save_dir  = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
err_path  = f"{save_dir}/{run_tag}_errors.csv"

rows_done = []
done_idx  = set()
if os.path.exists(pred_path):
    old = pd.read_csv(pred_path)
    if "row_idx" in old.columns:
        rows_done = old.to_dict("records")
        done_idx  = set(old["row_idx"].tolist())
        print(f"[resume] loaded {len(done_idx)} completed rows.")

err_rows = []
buf = []
save_every = 50

total = len(ds)
for i in tqdm(range(total)):
    if i in done_idx:
        continue
    x = ds[i]
    text = x["text"]
    gold = x["answer"]

    try:
        pred = ask_claude_textjson(text, LABELS)
        raw  = json.dumps({"label": pred})
    except Exception as e:
        pred = "UNKNOWN"
        raw  = f"ERROR: {type(e).__name__}: {e}"
        err_rows.append({
            "row_idx": i,
            "id": x.get("id", i),
            "error": raw,
            "text": text,
        })

    buf.append({
        "row_idx": i,
        "id": x.get("id", i),
        "text": text,
        "pred_raw": raw,
        "pred": pred,
        "label": gold,
    })

    if len(buf) % save_every == 0:
        out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
        out.to_csv(pred_path, index=False)
        if err_rows:
            pd.DataFrame(err_rows).to_csv(err_path, index=False)
        print(f"[checkpoint] saved {len(out)}/{total} -> {pred_path}")

out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
out.to_csv(pred_path, index=False)
if err_rows:
    pd.DataFrame(err_rows).to_csv(err_path, index=False)

print(f"[done] saved -> {pred_path}")
if os.path.exists(err_path):
    print(f"[errors] logged -> {err_path}")


In [ ]:
# Cell 6: Evaluate F1 / accuracy

import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

df = pd.read_csv(pred_path).sort_values("row_idx").drop_duplicates("row_idx", keep="last")
ok = df[df["pred"] != "UNKNOWN"].copy()

try:
    LABELS
except NameError:
    LABELS = ["negative", "neutral", "positive"]

f1ma = f1_score(ok["label"], ok["pred"], labels=LABELS, average="macro", zero_division=0)
acc  = accuracy_score(ok["label"], ok["pred"])

print(f"F1: {f1ma:.4f}, Accuracy: {acc:.4f}")
